# E4 — Decoder Instillation (Phase 10, W-lane flight)

**What this does**: ports the E1 recipe — the dictionary's angular contract as a
training target plus a retention channel — from the 22M embedder to a
generative model (Qwen2.5-1.5B-Instruct), via LoRA on a T4.

- **Geometry loss**: MSE between pooled-representation cosines (mean-pooled
  hidden states at `LOSS_LAYER`) and the grounded 14D target cosines, on the
  pack's train relations (80% split).
- **Retention channel**: chunked KL distillation toward the SAME model with
  adapters disabled (teacher = base, no second copy) on generic text — the
  E1 design fact that held semantic cost at zero, ported.
- **Arms**: real geometry vs `ARM_SCRAMBLED` marker → seeded derangement of
  the train targets (E6's control, built in from the start).
- **Evals pre/post**: held-out relation angular error by type · held-out
  random-pair r · ridge probe R² (reps→14D) · wikitext perplexity ·
  post-training per-layer probe sweep.

Recipe difference from E1, named: E1 trained all embedder parameters; E4
trains LoRA adapters (T4 constraint). `LOSS_LAYER` default 14 (mid-stack) —
E0 showed no layer strongly carries the geometry pre-training (held-out R²
0.13–0.32, flat), so the choice is about where instillation composes with
computation; the post-sweep adjudicates empirically.

**Pre-registered success**: held-out complement+synonym angular error drops
materially below baseline; random-pair r rises; perplexity within +5% of
base; probe R² at `LOSS_LAYER` rises. **Kill**: perplexity degrades >10% or
the geometry does not move — then the E1 recipe does not port at LoRA rank
and the program narrows per the prospectus.

SMOKE marker → 60 steps + subset evals (toolchain shakeout, not data).
Pack: `e4_dictionary_pack.json` (uploaded beside; includes wing v0 and the
scrambled targets). Pre-registration of record: this cell, locked at first
full flight.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import subprocess, sys, os, json, re, math, time, random
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Installing packages...')
# stale torchao on the Colab VM trips transformers' integration version check (smoke 1)
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','datasets','accelerate','scipy','pandas','scikit-learn'], check=True)

import torch
import numpy as np
assert torch.cuda.is_available(), 'No GPU — request a T4.'
DEV = 'cuda'

if subprocess.run(['which','rclone'], capture_output=True).returncode != 0:
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True, capture_output=True)
RCLONE_CONF = '/content/rclone.conf'
HAS_RCLONE = os.path.exists(RCLONE_CONF)

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists() and HAS_RCLONE:
    subprocess.run(['rclone','--config',RCLONE_CONF,'copy',
                    'gdrive:semcore/e4/e4_dictionary_pack.json','/content/'], check=True)
pack = json.load(open(PACK))
print('pack:', pack['name'], 'v'+pack['version'], '| concepts', pack['n_concepts'])

SMOKE = Path('/content/SMOKE').exists()
SCRAMBLED = Path('/content/ARM_SCRAMBLED').exists()
ARM = 'scrambled' if SCRAMBLED else 'real'
print('MODE:', 'SMOKE' if SMOKE else 'FULL', '| ARM:', ARM)

SEED = 20260821
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
LOSS_LAYER = 14
STEPS = 60 if SMOKE else 800
GEO_BS = 24          # pairs per geometry step (48 texts)
RET_BS, RET_LEN = 6, 256
LAMBDA_RET = 1.0
LR = 1e-4
OUT = Path('/content/e4_out'); OUT.mkdir(exist_ok=True)


Installing packages...


pack: E4 dictionary pack v1.0 | concepts 3052
MODE: FULL | ARM: scrambled


In [ ]:
# ── Pack tensors ─────────────────────────────────────────────────────────────
concepts = pack['concepts']
names = [c['name'] for c in concepts]
texts = [f"{c['name']}: {c['desc']}" if c['desc'] else c['name'] for c in concepts]
V14 = np.array([c['vec'] for c in concepts], float)
V14n = V14 / (np.linalg.norm(V14, axis=1, keepdims=True) + 1e-12)
idx_of = {n: i for i, n in enumerate(names)}

train_rels = pack['relations_train']
heldout_rels = pack['relations_heldout']
rand_pairs = pack['random_pairs_heldout']
train_targets = [r['angle14'] for r in train_rels]
if SCRAMBLED:
    train_targets = pack['scrambled_train_targets']
    print('ARM=scrambled: train targets deranged (seeded)')

train_items = [(idx_of[r['a']], idx_of[r['b']], math.cos(math.radians(t)))
               for r, t in zip(train_rels, train_targets)]

if SMOKE:
    ho_rels = heldout_rels[:300]
    ho_rand = rand_pairs[:500]
    probe_n = 800
else:
    ho_rels, ho_rand, probe_n = heldout_rels, rand_pairs, len(names)
print(f'train pairs {len(train_items)} | heldout rels {len(ho_rels)} | heldout random {len(ho_rand)}')


ARM=scrambled: train targets deranged (seeded)
train pairs 4213 | heldout rels 1058 | heldout random 2000


In [ ]:
# ── Model + LoRA + pooling ───────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEV)
lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'],
                  task_type='CAUSAL_LM')
model = get_peft_model(base, lcfg)
model.print_trainable_parameters()

ENC = tok(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')

def pooled(indices, grad=False, layer=LOSS_LAYER):
    ids = ENC.input_ids[indices].to(DEV)
    mask = ENC.attention_mask[indices].to(DEV)
    ctx = torch.enable_grad() if grad else torch.no_grad()
    with ctx:
        out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
        h = out.hidden_states[layer]
        m = mask.unsqueeze(-1).to(h.dtype)
        rep = (h * m).sum(1) / m.sum(1).clamp(min=1)
    return rep

def all_pooled(layer=LOSS_LAYER, bs=64, n=None):
    model.eval()
    reps = []
    N = n or len(names)
    with torch.no_grad():
        for i in range(0, N, bs):
            reps.append(pooled(torch.arange(i, min(i+bs, N)), layer=layer).float().cpu())
    return torch.cat(reps).numpy()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
# ── Eval suite ───────────────────────────────────────────────────────────────
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr

def angles_from_reps(reps, pairs):
    out = []
    for a, b in pairs:
        va, vb = reps[a], reps[b]
        c = float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-12))
        out.append(math.degrees(math.acos(max(-1, min(1, c)))))
    return np.array(out)

def geom_eval(reps):
    res = {}
    by_type = {}
    for r in ho_rels:
        by_type.setdefault(r['type'], []).append(r)
    for t, lst in sorted(by_type.items()):
        pairs = [(idx_of[r['a']], idx_of[r['b']]) for r in lst]
        model_ang = angles_from_reps(reps, pairs)
        target = np.array([r['angle14'] for r in lst])
        res[f'heldout_{t}_err_deg'] = round(float(np.abs(model_ang - target).mean()), 2)
    pairs = [(idx_of[r['a']], idx_of[r['b']]) for r in ho_rand]
    model_ang = angles_from_reps(reps, pairs)
    target = np.array([r['angle14'] for r in ho_rand])
    res['random_pair_r'] = round(float(pearsonr(model_ang, target).statistic), 4)
    ridx = np.random.default_rng(SEED).permutation(min(probe_n, len(names)))
    cut = int(0.8 * len(ridx))
    X, Y = reps[ridx], V14[ridx]
    reg = Ridge(alpha=1.0).fit(X[:cut], Y[:cut])
    ss_res = ((Y[cut:] - reg.predict(X[cut:]))**2).sum()
    ss_tot = ((Y[cut:] - Y[cut:].mean(0))**2).sum()
    res['probe_r2'] = round(float(1 - ss_res/ss_tot), 4)
    return res

from datasets import load_dataset
wt_test = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
wt_text = '\n\n'.join(t for t in wt_test['text'] if t.strip())[:60000 if SMOKE else 200000]

def perplexity():
    model.eval()
    ids = tok(wt_text, return_tensors='pt').input_ids[0]
    stride, seqlen = 512, 1024
    nlls, count = [], 0
    with torch.no_grad():
        for i in range(0, ids.size(0) - 1, stride):
            chunk = ids[i:i+seqlen].unsqueeze(0).to(DEV)
            if chunk.size(1) < 32: break
            out = model(chunk, labels=chunk)
            n = chunk.size(1) - 1
            nlls.append(float(out.loss) * n); count += n
    return round(math.exp(sum(nlls) / count), 4)

wt_train = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
wt_train_text = '\n\n'.join(t for t in wt_train['text'] if t.strip())
wt_ids = tok(wt_train_text[:2_000_000], return_tensors='pt').input_ids[0]

def retention_batch():
    starts = np.random.randint(0, wt_ids.size(0) - RET_LEN - 1, RET_BS)
    return torch.stack([wt_ids[s:s+RET_LEN] for s in starts]).to(DEV)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (461324 > 131072). Running this sequence through the model will result in indexing errors


In [ ]:
# ── Baseline eval (adapters disabled = the base model) ───────────────────────
with model.disable_adapter():
    base_reps = all_pooled()   # full dictionary — heldout pairs index all concepts (smoke-2 fix)
    baseline = geom_eval(base_reps)
    baseline['perplexity'] = perplexity()
print('BASELINE:', json.dumps(baseline, indent=1))


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.186760577644776e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


BASELINE: {
 "heldout_adjacent_err_deg": 50.77,
 "heldout_affinity_err_deg": 28.29,
 "heldout_complement_err_deg": 60.6,
 "heldout_homonym_err_deg": 41.55,
 "heldout_isomorphic_err_deg": 29.37,
 "heldout_opposition_err_deg": 74.2,
 "heldout_synonym_err_deg": 14.8,
 "random_pair_r": -0.0152,
 "probe_r2": -0.5714,
 "perplexity": 10.658
}


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
def kl_chunked(student_logits, teacher_logits, chunk=512):
    # KL(teacher || student) summed over positions, chunked over the flattened batch
    s = student_logits.reshape(-1, student_logits.size(-1))
    t = teacher_logits.reshape(-1, teacher_logits.size(-1))
    total, n = 0.0, s.size(0)
    for i in range(0, n, chunk):
        sl = torch.log_softmax(s[i:i+chunk].float(), -1)
        tl = torch.softmax(t[i:i+chunk].float(), -1)
        total = total + (tl * (torch.log(tl + 1e-9) - sl)).sum()
    return total / n

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler('cuda')
model.train()
t0 = time.time()
log = []
for step in range(1, STEPS + 1):
    opt.zero_grad()
    # geometry loss
    batch = random.sample(train_items, GEO_BS)
    ia = torch.tensor([b[0] for b in batch]); ib = torch.tensor([b[1] for b in batch])
    tgt = torch.tensor([b[2] for b in batch], device=DEV, dtype=torch.float32)
    with torch.amp.autocast('cuda'):
        ra = pooled(ia, grad=True); rb = pooled(ib, grad=True)
        cos = torch.nn.functional.cosine_similarity(ra.float(), rb.float())
        l_geo = torch.nn.functional.mse_loss(cos, tgt)
        # retention: KL toward the frozen base on generic text
        rb_ids = retention_batch()
        with model.disable_adapter(), torch.no_grad():
            t_logits = model(rb_ids).logits
        s_logits = model(rb_ids).logits
        l_ret = kl_chunked(s_logits, t_logits)
        loss = l_geo + LAMBDA_RET * l_ret
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    scaler.step(opt); scaler.update()
    if step % 20 == 0 or step == 1:
        rec = dict(step=step, geo=round(float(l_geo.detach()),4), ret=round(float(l_ret.detach()),4),
                   elapsed=round(time.time()-t0,1))
        log.append(rec); print(rec)
print(f'training done in {time.time()-t0:.0f}s')


{'step': 1, 'geo': 0.1384, 'ret': 0.0001, 'elapsed': 1.9}


{'step': 20, 'geo': 0.1136, 'ret': 0.0001, 'elapsed': 38.3}


{'step': 40, 'geo': 0.07, 'ret': 0.0002, 'elapsed': 77.3}


{'step': 60, 'geo': 0.1069, 'ret': 0.0012, 'elapsed': 115.9}


{'step': 80, 'geo': 0.1416, 'ret': 0.0002, 'elapsed': 154.8}


...

{'step': 720, 'geo': 0.0743, 'ret': 0.0025, 'elapsed': 1397.2}


{'step': 740, 'geo': 0.0463, 'ret': 0.0027, 'elapsed': 1435.9}


{'step': 760, 'geo': 0.0407, 'ret': 0.0029, 'elapsed': 1474.5}


{'step': 780, 'geo': 0.0318, 'ret': 0.0026, 'elapsed': 1513.3}


{'step': 800, 'geo': 0.0391, 'ret': 0.0022, 'elapsed': 1552.1}
training done in 1552s


In [ ]:
# ── Post eval + layer sweep + ship ───────────────────────────────────────────
post_reps = all_pooled()
post = geom_eval(post_reps)
post['perplexity'] = perplexity()
print('POST:', json.dumps(post, indent=1))

sweep = {}
if not SMOKE:
    for L in range(0, base.config.num_hidden_layers + 1, 2):
        reps_L = all_pooled(layer=L)
        ridx = np.random.default_rng(SEED).permutation(len(names))
        cut = int(0.8 * len(ridx))
        from sklearn.linear_model import Ridge
        reg = Ridge(alpha=1.0).fit(reps_L[ridx[:cut]], V14[ridx[:cut]])
        ss_res = ((V14[ridx[cut:]] - reg.predict(reps_L[ridx[cut:]]))**2).sum()
        ss_tot = ((V14[ridx[cut:]] - V14[ridx[cut:]].mean(0))**2).sum()
        sweep[str(L)] = round(float(1 - ss_res/ss_tot), 4)
        print(f'layer {L}: probe R2 {sweep[str(L)]}')

import datetime
verdict = {
    'flight': f'E4 {ARM} ' + ('SMOKE' if SMOKE else 'FULL'),
    'date': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': MODEL_ID, 'loss_layer': LOSS_LAYER, 'steps': STEPS,
    'lora': 'r16 a32 qkvo', 'lambda_ret': LAMBDA_RET,
    'baseline': baseline, 'post': post, 'layer_sweep_post': sweep,
    'delta': {k: round(post[k] - baseline[k], 4) for k in baseline if isinstance(baseline[k], (int, float))},
    'train_log': log,
}
(OUT / 'e4_verdict.json').write_text(json.dumps(verdict, indent=1))
print(json.dumps({k: v for k, v in verdict.items() if k not in ('train_log','layer_sweep_post')}, indent=1))

model.save_pretrained(str(OUT / f'adapter_{ARM}'))
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
if HAS_RCLONE:
    subprocess.run(['rclone','--config',RCLONE_CONF,'copy',str(OUT),
                    f'gdrive:semcore/e4/{ARM}_{"smoke" if SMOKE else "full"}_{stamp}'], check=True)
    print('shipped to', f'gdrive:semcore/e4/{ARM}_{"smoke" if SMOKE else "full"}_{stamp}')


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.5237146655522338e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


POST: {
 "heldout_adjacent_err_deg": 17.21,
 "heldout_affinity_err_deg": 16.96,
 "heldout_complement_err_deg": 23.77,
 "heldout_homonym_err_deg": 16.67,
 "heldout_isomorphic_err_deg": 15.72,
 "heldout_opposition_err_deg": 25.83,
 "heldout_synonym_err_deg": 22.27,
 "random_pair_r": -0.0023,
 "probe_r2": -0.7634,
 "perplexity": 10.6614
}


layer 0: probe R2 0.2181


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.338875380151876e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


layer 2: probe R2 -0.4082


...

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 8.102112758479052e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


layer 26: probe R2 -0.8011


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.7151381587486867e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


layer 28: probe R2 -0.8605
{
 "flight": "E4 scrambled FULL",
 "date": "2026-08-21T22:21:01.701619+00:00",
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "loss_layer": 14,
 "steps": 800,
 "lora": "r16 a32 qkvo",
 "lambda_ret": 1.0,
 "baseline": {
  "heldout_adjacent_err_deg": 50.77,
  "heldout_affinity_err_deg": 28.29,
  "heldout_complement_err_deg": 60.6,
  "heldout_homonym_err_deg": 41.55,
  "heldout_isomorphic_err_deg": 29.37,
  "heldout_opposition_err_deg": 74.2,
  "heldout_synonym_err_deg": 14.8,
  "random_pair_r": -0.0152,
  "probe_r2": -0.5714,
  "perplexity": 10.658
 },
 "post": {
  "heldout_adjacent_err_deg": 17.21,
  "heldout_affinity_err_deg": 16.96,
  "heldout_complement_err_deg": 23.77,
  "heldout_homonym_err_deg": 16.67,
  "heldout_isomorphic_err_deg": 15.72,
  "heldout_opposition_err_deg": 25.83,
  "heldout_synonym_err_deg": 22.27,
  "random_pair_r": -0.0023,
  "probe_r2": -0.7634,
  "perplexity": 10.6614
 },
 "delta": {
  "heldout_adjacent_err_deg": -33.56,
  "heldout_affinity

shipped to gdrive:semcore/e4/scrambled_full_20260821_2221
